# M4 component profiling: T4 smoke first
Thin launcher for the repository CLI. No measurement formulas live here.
The first cells inspect actual GPU/driver before downloads or installation.
No canonical topology profile or gamma is produced. See `experiments/M4_COLAB.md`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, json, re
EXPECTED_GPU = "T4"
RUN_ID = "CHANGE-ME-unique-t4-smoke"
REPO_URL = "https://github.com/theflyingkhui04/do-an-ptit-d22.git"
BRANCH = "feat/m4-02-02-gpu-cost-profiler"
REVISION = "SET-TO-REVIEWED-40-CHAR-COMMIT"
REPO = Path("do-an-ptit-d22")
assert RUN_ID != "CHANGE-ME-unique-t4-smoke", "Choose a unique run ID before inspection"
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", RUN_ID) and ".." not in RUN_ID, "Invalid run ID"
inspection = dict(run_id=RUN_ID, run_type="smoke", artifact_kind="hardware_preflight",
                  benchmark_result=False, model_loaded=False,
                  timestamp_utc=datetime.now(timezone.utc).isoformat(),
                  expected_gpu=EXPECTED_GPU, observed_gpu=None, gpu=None,
                  gpu_compute_capability=None, sm_architecture=None,
                  gpu_memory_total_mb=None, driver_version=None, cuda_version=None,
                  torch_version=None, source_commit=None)
try:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=20)
    print(smi.stdout)
    inspection.update(nvidia_smi_output=smi.stdout, nvidia_smi_stderr=smi.stderr,
                      nvidia_smi_returncode=smi.returncode)
    driver_query = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                                  capture_output=True, text=True, timeout=20)
    if driver_query.returncode == 0 and driver_query.stdout.strip():
        inspection["driver_version"] = driver_query.stdout.splitlines()[0].strip()
except (OSError, subprocess.TimeoutExpired) as exc:
    inspection["nvidia_smi_error"] = str(exc)
try:
    import torch  # Colab kernel is used only for this early inspection.
    inspection.update(torch_version=torch.__version__, cuda_version=torch.version.cuda)
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        capability = tuple(torch.cuda.get_device_capability(0))
        inspection.update(observed_gpu=name, gpu=name,
                          gpu_compute_capability=list(capability),
                          sm_architecture=f"sm_{capability[0]}{capability[1]}",
                          gpu_memory_total_mb=torch.cuda.get_device_properties(0).total_memory / 2**20)
        print(name, capability)
except Exception as exc:
    inspection["torch_inspection_error"] = f"{type(exc).__name__}: {exc}"
# Before checkout, use the current directory as the repository-relative evidence root.
# Do not create an empty REPO directory that would interfere with a later clone.
EVIDENCE_ROOT = REPO if (REPO / ".git").exists() else Path(".")
if (EVIDENCE_ROOT / ".git").exists():
    git_sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=EVIDENCE_ROOT,
                             capture_output=True, text=True)
    if git_sha.returncode == 0:
        inspection["source_commit"] = git_sha.stdout.strip()
expected_cap = {"T4": [7,5], "L4": [8,9]}[EXPECTED_GPU]
hardware_matches = (EXPECTED_GPU in (inspection["gpu"] or "").split() and
                    inspection["gpu_compute_capability"] == expected_cap)
inspection["canonical_hardware"] = hardware_matches

def retain_preflight(status):
    details = dict(inspection, status=status)
    out = EVIDENCE_ROOT / "experiments" / "runs" / "m4_cost_profiling" / RUN_ID
    out.mkdir(parents=True, exist_ok=False)  # Never replace existing run evidence.
    for filename in ("env.json", "hardware_inspection.json"):
        (out / filename).write_text(json.dumps(details, indent=2) + "\n", encoding="utf-8")
    (out / "nvidia-smi.txt").write_text(inspection.get("nvidia_smi_output", ""), encoding="utf-8")
    print("Hardware preflight evidence retained:", out.resolve())

if not hardware_matches:
    retain_preflight("hardware_mismatch")
    raise SystemExit("GPU mismatch: non-canonical hardware evidence only; stop before installation/model downloads")
driver = inspection["driver_version"]
if driver is None or int(driver.split(".")[0]) < 580:
    retain_preflight("driver_incompatible")
    raise SystemExit("Pinned CUDA 13 stack requires a compatible driver; stop before installation/model downloads")


In [ ]:
assert len(REVISION) == 40 and all(c in "0123456789abcdef" for c in REVISION), "Set the reviewed immutable commit"
if not REPO.exists():
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)])
# Checkout refuses conflicting local changes; no reset or clean commands.
subprocess.check_call(["git", "checkout", REVISION], cwd=REPO)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True))


In [ ]:
# Explicit CUDA wheel installation in an isolated interpreter.
subprocess.check_call([sys.executable, "-m", "venv", ".venv"], cwd=REPO)
PYTHON = REPO.resolve() / ".venv" / "bin" / "python"
subprocess.check_call([str(PYTHON), "-m", "pip", "install", "--index-url",
                       "https://download.pytorch.org/whl/cu130", "torch==2.13.0+cu130"], cwd=REPO)
subprocess.check_call([str(PYTHON), "-m", "pip", "install", "--index-url", "https://pypi.org/simple",
                       "-r", "experiments/requirements-m4-cuda.txt"], cwd=REPO)
# Retain setup evidence separately so the CLI can reserve its normal run directory.
SETUP = REPO / "experiments" / "runs" / "m4_cost_profiling_setup" / RUN_ID
SETUP.mkdir(parents=True, exist_ok=False)
VERSION_CHECK = r'''
import json, sys
from pathlib import Path
import torch, vllm
print(torch.__version__)
print(torch.version.cuda)
print(vllm.__version__)
versions = dict(torch_version=torch.__version__, cuda_version=torch.version.cuda,
                vllm_version=vllm.__version__, python_executable=sys.executable,
                torch_file=torch.__file__, vllm_file=vllm.__file__)
versions["versions_match"] = (versions["torch_version"] == "2.13.0+cu130" and
                              versions["cuda_version"] == "13.0" and
                              versions["vllm_version"] == "0.30.0")
Path(sys.argv[1]).write_text(json.dumps(versions, indent=2) + "\n", encoding="utf-8")
assert versions["torch_version"] == "2.13.0+cu130", "Wrong PyTorch release/build"
assert versions["cuda_version"] == "13.0", "Wrong PyTorch CUDA runtime"
assert versions["vllm_version"] == "0.30.0", "Wrong vLLM release"
'''
checked = subprocess.run([str(PYTHON), "-c", VERSION_CHECK, str((SETUP / "software_versions.json").resolve())],
                         cwd=REPO, capture_output=True, text=True)
(SETUP / "version_check.stdout.txt").write_text(checked.stdout, encoding="utf-8")
(SETUP / "version_check.stderr.txt").write_text(checked.stderr, encoding="utf-8")
print(checked.stdout)
print(checked.stderr, file=sys.stderr)
checked.check_returncode()  # Version/import failure must stop before the smoke cell.
subprocess.check_call([str(PYTHON), "-m", "pip", "check"], cwd=REPO)


In [ ]:
assert checked.returncode == 0 and json.loads((SETUP / "software_versions.json").read_text())["versions_match"], "Pass isolated version checks first"
assert RUN_ID != "CHANGE-ME-unique-t4-smoke", "Choose a unique run ID"
command = [str(PYTHON), "-m", "experiments.m4_cost_runner", "--run-id", RUN_ID,
           "--expected-gpu", EXPECTED_GPU, "--mode", "smoke", "--contexts", "256",
           "--batches", "1", "--k-values", "1", "--warmup", "2", "--repeats", "2"]
completed = subprocess.run(command, cwd=REPO)
print("CLI exit:", completed.returncode)

# Include the setup version proof in the smoke export without changing profiler behavior.
ARTIFACTS = REPO / "experiments" / "runs" / "m4_cost_profiling" / RUN_ID
if ARTIFACTS.exists():
    with (ARTIFACTS / "setup_software_versions.json").open("x", encoding="utf-8") as proof:
        proof.write((SETUP / "software_versions.json").read_text(encoding="utf-8"))


In [ ]:
ARTIFACTS = REPO / "experiments" / "runs" / "m4_cost_profiling" / RUN_ID
for filename in ("env.json", "run_manifest.json", "component_report.json"):
    path = ARTIFACTS / filename
    if path.exists():
        print(filename, path.read_text())
import shutil
archive = shutil.make_archive("m4-" + RUN_ID, "zip", root_dir=ARTIFACTS.parent, base_dir=RUN_ID)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Artifacts archive:", archive)


This smoke is not benchmark evidence. Keep every artifact, including errors and warmups.
For component-campaign commands after compatibility validation, use M4_COLAB.md.
Do not generate canonical profiles until both cost/provenance alignment and tree-runtime/model approval are resolved.
